# Lab — Images as Data: Region of Interest to Average Color
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

The first step of vision analytics is turning an image into numbers. Pick a **Region of Interest (ROI)**, average its pixels, and a part becomes a **feature vector** you can later classify and cluster.

> **Skeleton:** this notebook generates a synthetic image so it runs anywhere. Swap in a real image (`base.jpg`, the Training Data folders) where marked.

## Objectives
By the end of this lab you will be able to:
- See an image as a NumPy array of pixels (OpenCV uses **BGR** order).
- Define ROIs as `(name, x, y, w, h)`.
- Compute the **average RGB** of each ROI — an image turned into a feature vector.
- Read the result as a data table: the input to classification (Lab 4) and clustering (Lab 5).

## Setup
OpenCV + NumPy + matplotlib. Note: `cv2.imshow` does not work in Colab, so we display with matplotlib.

In [ ]:
!pip -q install opencv-python-headless

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
print("cv2", cv2.__version__)

## 1 · Get an image
We build a synthetic "station" image with coloured parts so the lab runs anywhere and is *self-checking* — the measured colour should match the colour we drew.

**To use your own image:** replace this cell with `img = cv2.imread("your_image.jpg")` (e.g. `base.jpg` or a Training Data image from the vision toolkit).

In [ ]:
# OpenCV images are BGR. Dark-grey background with four coloured swatches.
img = np.full((300, 500, 3), 60, dtype=np.uint8)
swatches = {
    (40,  40, 100, 100): (40, 40, 200),   # red-ish  (B,G,R)
    (200, 40, 100, 100): (200, 60, 40),   # blue-ish
    (360, 40, 100, 100): (30, 30, 30),    # near-black
    (200, 180, 100, 80): (90, 160, 90),   # 'noise' / green
}
for (x, y, w, h), color in swatches.items():
    img[y:y+h, x:x+w] = color

def show(bgr, title=""):
    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

show(img, "Synthetic station image")

## 2 · Define Regions of Interest
Same convention as the vision toolkit: `(name, x, y, w, h)`.

In [ ]:
ROIs = [
    ("part_1", 40,  40, 100, 100),
    ("part_2", 200, 40, 100, 100),
    ("part_3", 360, 40, 100, 100),
    ("part_4", 200, 180, 100, 80),
]

overlay = img.copy()
for name, x, y, w, h in ROIs:
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (255, 255, 255), 2)
    cv2.putText(overlay, name, (x, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
show(overlay, "Regions of Interest")

## 3 · Measure average colour (image to feature vector)
Slice the ROI out of the array and average all its pixels. OpenCV stores BGR; we report **RGB**.

In [ ]:
def average_color(image, roi):
    name, x, y, w, h = roi
    patch = image[y:y+h, x:x+w]        # (h, w, 3), BGR
    b, g, r = patch.mean(axis=(0, 1))  # mean over every pixel in the ROI
    return round(r, 1), round(g, 1), round(b, 1)

readings = []
for roi in ROIs:
    r, g, b = average_color(img, roi)
    readings.append({"roi": roi[0], "R": r, "G": g, "B": b})
    print(f"{roi[0]:8}  R={r:6.1f}  G={g:6.1f}  B={b:6.1f}")

## 4 · Read it as data
Each ROI is now a row of `(R, G, B)` — a **feature vector**. This table is exactly the input to classification (Lab 4) and clustering (Lab 5).

In [ ]:
df = pd.DataFrame(readings)
df

## Debrief
- Each part is now just 3 numbers. Why is that enough to tell Red from Blue, but maybe *not* Red from a slightly-faded Red?
- The "noise"/green swatch: how would you flag it as **not** one of your known classes?
- What happens to these numbers if the lighting changes? (See the Data-Quality session — "like colours brighten, opposite colours darken".)

_Next: **Lab 3** builds a labelled reference from many samples; **Lab 4** classifies a new part; **Lab 5** clusters these vectors in 3-D RGB space._

_TODO (next run): swap the synthetic image for `base.jpg` / the `Training Data` folders and re-run._